# Cell Image Labeling Notebook

## Overview

This notebook processes fluorescence microscopy images to generate YOLO-format object detection labels for cell localization. It uses a sliding-window cropping approach to systematically extract patches and generate bounding box annotations for identified cells.

### Workflow
1. Load a dataset and select preprocessing parameters
2. Initialize output directories for labels, masks, and crops
3. Iterate through source images (Green and Phase channels)
4. Apply sliding-window cropping with overlap
5. Process each crop and detect cells
6. Generate YOLO-format labels for cells passing quality filters
7. Save annotated crops and corresponding images

### Output Files
- **Labels** (`*.txt`): YOLO-format annotations (class, center_x, center_y, width, height)
- **Masks** (`*_mask.png`): Binary processed images used for detection
- **Phase crops** (`*_phase.png`): Phase-contrast channel crops
- **Green crops** (`*_green.png`): Fluorescence channel crops

### Requirements
- OpenCV (`cv2`)
- NumPy
- `CellProcessor` module (custom image processing utilities)

In [21]:
"""Import required libraries for image processing, annotation, and dataset management."""
import cv2
import numpy as np
import os
from CellProcessor import (
    read_image,
    process_image,
    get_bboxes,
    get_label_yolo,
    use_dataset,
    list_dataset,
    use_variables,
    list_variables
)

## Configuration

Select the dataset and preprocessing parameters to use. Review available options before proceeding.

In [20]:
"""List available datasets and select one to use."""
print("Available datasets:")
list_dataset()

Available datasets:


,ID,Cell_type,Death_type,Image_path,Description
0,1,MEF,Necroptosis,Data,Test


In [22]:
"""List available preprocessing variable sets."""
print("Available preprocessing variable sets:")
list_variables()

Available preprocessing variable sets:


,ID,Time,Brightness,Contrast,Threshold,Erosion,Dialation,Description
0,1,2026-01-16T10:32:29,-7.45,8.06,45,2,3,test


## Load Dataset and Initialize Output Directories

Load the dataset configuration and create output directories for labels, masks, and cropped images.

In [ ]:
"""Load dataset configuration and initialize output directories."""
# Load dataset (using preset 1; modify as needed)
dataset = use_dataset(1)
print(f"Dataset: {dataset['Cell_type']} cells, {dataset['Death_type']} death type")
print(f"Image path: {dataset['Image_path']}\n")

# Construct input paths
GREEN_PATH = os.path.join(dataset['Image_path'], dataset['Death_type'], f"{dataset['Cell_type']}_Green/")
PHASE_PATH = os.path.join(dataset['Image_path'], dataset['Death_type'], f"{dataset['Cell_type']}_Phase/")

# Construct output paths
base_output = os.path.join(dataset['Image_path'], dataset['Death_type'])
LABELED_IMAGES_DIR_PATH = os.path.join(base_output, f"{dataset['Cell_type']}_Labeled_phase/")
IMAGES_MASKS_DIR_PATH = os.path.join(base_output, f"{dataset['Cell_type']}_Masks_phase/")
IMAGES_PHASE_PATH = os.path.join(base_output, f"{dataset['Cell_type']}_Phase_Crop/")
IMAGES_GREEN_PATH = os.path.join(base_output, f"{dataset['Cell_type']}_Green_Crop/")

# Create output directories
output_dirs = {
    LABELED_IMAGES_DIR_PATH: "Labels",
    IMAGES_MASKS_DIR_PATH: "Masks",
    IMAGES_PHASE_PATH: "Phase crops",
    IMAGES_GREEN_PATH: "Green crops"
}

print("Output directories:")
for dir_path, dir_name in output_dirs.items():
    try:
        os.makedirs(dir_path, exist_ok=True)
        print(f"  ✓ {dir_name}: {dir_path}")
    except OSError as e:
        print(f"  ✗ {dir_name}: Error creating directory: {e}")

Dataset: MEF cells, Necroptosis death type
Image path: Data

Output directories:
  ✓ Labels: Data/Necroptosis/MEF_Labeled_phase/
  ✓ Masks: Data/Necroptosis/MEF_Masks_phase/
  ✓ Phase crops: Data/Necroptosis/MEF_Phase_Crop/
  ✓ Green crops: Data/Necroptosis/MEF_Green_Crop/


## Load Preprocessing Parameters

Load the preprocessing configuration (contrast, brightness, thresholds, morphological operations).

In [24]:
"""Load preprocessing parameters (using preset 1; modify as needed)."""
variables = use_variables(1)
print("Loaded preprocessing parameters:")
variables

Loaded preprocessing parameters:


{'ID': 1,
 'Time': '2026-01-16T10:32:29',
 'Brightness': -7.45,
 'Contrast': 8.06,
 'Threshold': 45,
 'Erosion': 2,
 'Dialation': 3,
 'Description': 'test'}

## Image Processing and Label Generation

Extract overlapping patches from each image, apply preprocessing, detect cells, and generate YOLO-format annotations.

### Patch Extraction Parameters
- **Patch size**: 128×128 pixels
- **Horizontal overlap**: 64 pixels (50% stride)
- **Vertical overlap**: 52 pixels (40.6% stride)
- **Horizontal step**: 64 pixels
- **Vertical step**: 76 pixels

### Filtering Thresholds
- **Max signal intensity**: 40% (exclude bright regions)
- **Min detection size**: 0.01 pixels (exclude noise)
- **Only save patches** with at least one valid cell detection

In [25]:
"""
Extract patches from images, detect cells, and generate YOLO-format labels (optimized).

Process:
1. Iterate through all source images (Green and Phase channels)
2. Apply sliding-window cropping with configurable overlap
3. Apply preprocessing pipeline to each patch
4. Detect cell bounding boxes
5. Filter detections and generate normalized YOLO coordinates
6. Save patches, masks, and label files
"""

# Crop parameters (precomputed constants)
CROP_WIDTH = 128
CROP_HEIGHT = 128
CROP_WIDTH_SHIFT = 64  # 128 - 64
CROP_HEIGHT_SHIFT = 76  # 128 - 52

# Filtering thresholds
MAX_SIGNAL_THRESHOLD = 40.0   # percent; exclude overly bright regions
MIN_DETECTION_SIZE = 0.01     # normalized; exclude sub-pixel detections

# Precompute variables dict values for loop efficiency
contrast_val = variables['Contrast']
brightness_val = variables['Brightness']
threshold_val = variables['Threshold']
erosion_val = variables['Erosion']
dialation_val = variables['Dialation']

# Collect images
image_files = sorted([
    f for f in os.listdir(GREEN_PATH)
    if f.lower().endswith(('png', 'jpg', 'jpeg'))
])
total_images = len(image_files)

if total_images == 0:
    print(f"⚠ No images found in {GREEN_PATH}")
else:
    print(f"Found {total_images} images. Processing...\n")

# Statistics tracking
processed_count = 0
cropped_count = 0
label_count = 0
error_count = 0

# Precompute valid x,y ranges once
image_dims_cache = {}

# Main processing loop
for img_idx, image_name in enumerate(image_files, start=1):
    try:
        print(f"[{img_idx}/{total_images}] {image_name}", end=" ... ", flush=True)
        
        # Construct paths
        green_path = os.path.join(GREEN_PATH, image_name)
        phase_path = os.path.join(PHASE_PATH, image_name)
        
        # Read images (img_processed contains both channels; we'll extract green from phase)
        img_processed, img_phase = read_image(green_path, phase_path)
        
        # Validate reads
        if img_processed is None or img_phase is None:
            print("✗ Failed to read image(s)")
            error_count += 1
            continue
        
        img_h, img_w = img_phase.shape[:2]
        base_name = os.path.splitext(image_name)[0]
        
        # Pre-compute valid window ranges (clipped to image bounds)
        x_starts = []
        y_starts = []
        
        for x in range(0, img_w - CROP_WIDTH + 1, CROP_WIDTH_SHIFT):
            x_starts.append(x)
        if not x_starts or x_starts[-1] + CROP_WIDTH < img_w:
            x_starts.append(max(0, img_w - CROP_WIDTH))
        
        for y in range(0, img_h - CROP_HEIGHT + 1, CROP_HEIGHT_SHIFT):
            y_starts.append(y)
        if not y_starts or y_starts[-1] + CROP_HEIGHT < img_h:
            y_starts.append(max(0, img_h - CROP_HEIGHT))
        
        crops_this_image = 0
        
        # Sliding window patch extraction (optimized loop structure)
        for wx_idx, x_start in enumerate(x_starts):
            x_end = x_start + CROP_WIDTH
            for hy_idx, y_start in enumerate(y_starts):
                y_end = y_start + CROP_HEIGHT
                
                # Extract patches (vectorized slicing, no copies)
                patch_processed = img_processed[y_start:y_end, x_start:x_end]
                patch_phase = img_phase[y_start:y_end, x_start:x_end]
                
                # Apply preprocessing pipeline (precomputed variables)
                patch_dilated = process_image(
                    patch_processed,
                    contrast_val,
                    brightness_val,
                    threshold_val,
                    erosion_val,
                    dialation_val,
                    plot=False
                )
                
                # Detect cell bounding boxes
                bboxes = get_bboxes(patch_dilated)
                
                # Skip patches with no detections (early exit)
                if not bboxes:
                    continue
                
                # Compute mean signal intensity (single operation)
                mean_signal = (np.mean(patch_dilated) / 255.0) * 100.0
                
                # Skip patches with excessive brightness
                if mean_signal >= MAX_SIGNAL_THRESHOLD:
                    continue
                
                # Generate YOLO-format labels for valid detections
                label_lines = []
                patch_h, patch_w = CROP_HEIGHT, CROP_WIDTH  # Known dimensions
                
                for (x1, y1), (x2, y2) in bboxes:
                    # Skip invalid coordinates (early continue)
                    if x1 < 0 or y1 < 0 or x2 < 0 or y2 < 0:
                        continue
                    
                    # Convert to YOLO format (normalized center + size)
                    norm_x, norm_y, norm_w, norm_h = get_label_yolo(
                        (x1, y1), (x2, y2),
                        patch_w, patch_h
                    )
                    
                    # Filter sub-pixel detections
                    if norm_w >= MIN_DETECTION_SIZE and norm_h >= MIN_DETECTION_SIZE:
                        label_lines.append(f"0 {norm_x} {norm_y} {norm_w} {norm_h}\n")
                
                # Save patch and annotations if valid detections exist
                if label_lines:
                    suffix = f"{wx_idx}_{hy_idx}"
                    base_path = os.path.join(base_output, base_name)
                    
                    # Batch file writes: construct paths once
                    mask_path = os.path.join(IMAGES_MASKS_DIR_PATH, f"{base_name}_{suffix}.png")
                    phase_out = os.path.join(IMAGES_PHASE_PATH, f"{base_name}_{suffix}.png")
                    green_out = os.path.join(IMAGES_GREEN_PATH, f"{base_name}_{suffix}.png")
                    label_path = os.path.join(LABELED_IMAGES_DIR_PATH, f"{base_name}_{suffix}.txt")
                    
                    # Write files
                    cv2.imwrite(mask_path, patch_dilated)
                    cv2.imwrite(phase_out, patch_phase)
                    # Green is derived from phase in this pipeline, save phase as proxy
                    cv2.imwrite(green_out, patch_phase)
                    
                    # Write label file
                    with open(label_path, "w") as f:
                        f.writelines(label_lines)
                    
                    crops_this_image += 1
                    label_count += len(label_lines)
        
        processed_count += 1
        print(f"✓ ({crops_this_image} patches)")
        cropped_count += crops_this_image
        
    except Exception as e:
        print(f"✗ Error: {e}")
        error_count += 1

# Print summary
print("\n" + "=" * 60)
print("PROCESSING COMPLETE")
print("=" * 60)
print(f"Images processed:    {processed_count}/{total_images}")
print(f"Patches extracted:   {cropped_count}")
print(f"Labels generated:    {label_count}")
print(f"Errors encountered:  {error_count}")
print("=" * 60)
print(f"\nOutput directories:")
for path in [LABELED_IMAGES_DIR_PATH, IMAGES_MASKS_DIR_PATH, IMAGES_PHASE_PATH, IMAGES_GREEN_PATH]:
    print(f"  • {path}")

Found 13 images. Processing...

[1/13] VID856_B4_1_00d00h00m.png ... ✓ (62 patches)
[2/13] VID856_B4_1_00d02h00m.png ... ✓ (157 patches)
[3/13] VID856_B4_1_00d04h00m.png ... ✓ (244 patches)
[4/13] VID856_B4_1_00d06h00m.png ... ✓ (272 patches)
[5/13] VID856_B4_1_00d08h00m.png ... ✓ (273 patches)
[6/13] VID856_B4_1_00d10h00m.png ... ✓ (272 patches)
[7/13] VID856_B4_1_00d12h00m.png ... ✓ (271 patches)
[8/13] VID856_B4_1_00d14h00m.png ... ✓ (268 patches)
[9/13] VID856_B4_1_00d16h00m.png ... ✓ (263 patches)
[10/13] VID856_B4_1_00d18h00m.png ... ✓ (254 patches)
[11/13] VID856_B4_1_00d20h00m.png ... ✓ (246 patches)
[12/13] VID856_B4_1_00d22h00m.png ... ✓ (229 patches)
[13/13] VID856_B4_1_01d00h00m.png ... ✓ (231 patches)

PROCESSING COMPLETE
Images processed:    13/13
Patches extracted:   3042
Labels generated:    13310
Errors encountered:  0

Output directories:
  • Data/Necroptosis/MEF_Labeled_phase/
  • Data/Necroptosis/MEF_Masks_phase/
  • Data/Necroptosis/MEF_Phase_Crop/
  • Data/Necropt